# Build 1 - RAG Time
CS 497  
Spring 2026  
Dr. Henderson  
_v1_

---

The objective of this assignment is to give you practical experience with building a full functional RAG system.

### 1. Embeddings

The first step is to process the local files to use in the RAG engine. This includes chunking, embedding, and storing in a vector database. Regardless of which model provider you decide to use for generation, we will use a local embedding model so you will need to have `ollama` installed and the ollama server running.

1.1 (5 pts) Test the embedding model by importing the `OllamaEmbeddings` class from `langchain_ollama`, creating an instance using `model = 'mxbai-embed-large'`, and calling the `embed_query()` method on it with random sentence. Be sure to assign the result to a variable for the next question.

In [2]:
# https://reference.langchain.com/python/langchain-ollama/embeddings/OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings

embed = OllamaEmbeddings(model="mxbai-embed-large")

input_text = "The meaning of life is 42"
vector = embed.embed_query(input_text)

1.2 (2 pts) What is the type of the resulting query embedding (show using Python)?

In [3]:
type(vector)

list

1.3 (3 pts) What are the dimensions of the query embedding (show using Python)?

In [4]:
len(vector)

1024

1.4 (2 pts) Show the first 10 values of the query embedding.

In [5]:
vector[:10]

[0.014529221,
 -0.035248388,
 0.003272863,
 -0.024100352,
 -0.042415757,
 0.005856152,
 0.025113212,
 0.00019563432,
 0.054264843,
 0.0053503932]

1.5 (2 pts) Create another query embedding by replacing one (significant) word in your original query and show the first 10 values.

In [6]:
input_text2 = "The meaning of life is 67"
vector2 = embed.embed_query(input_text2)
vector[:10]

[0.014529221,
 -0.035248388,
 0.003272863,
 -0.024100352,
 -0.042415757,
 0.005856152,
 0.025113212,
 0.00019563432,
 0.054264843,
 0.0053503932]

1.6 (3 pts) Calculate the similarity between the first embedding _and itself_ using the dot product.

In [7]:
import numpy as np
np.dot(vector, vector)

np.float64(1.0000010940000625)

1.7 (3 pts) Calculate the similarity between the first and second embedding using the dot product.

In [8]:
np.dot(vector, vector2)

np.float64(0.8244128607850277)

### 2. Building the Database

We'll use the Chroma vector database to store the embeddings and implement the RAG server.

2.1 (5 pts) Import the `Chroma` class from `langchain_chroma` and create a new instance named `vector_store`. You will need to pass in the following parameters: `collection_name` (use something like "cs497_b1"), `embedding_function` (set this to the instance you created in 1.1), and `persist_directory` (the path where you want the database to live - the file will be called `chroma.sqlite3`).

In [9]:
from langchain_chroma import Chroma

vector_store = Chroma(collection_name='cs497_b1',
                      embedding_function=embed,
                      persist_directory='chroma_db')

To embed documents they should first be split up into predictable lengths. This helps the embeddings be more semantically granular and also means less text needs to be added to the context window of the model during generation.  We'll use a standard text splitter from langchain.

2.2 (5 pts) Import the `RecursiveCharacterTextSplitter` from the `langchain_text_splitters` module and create an instance named `text_splitter` with the following parameters: `chunk_size` (set to 1000), `chunk_overlap` (set to 200), `length_function` (set to `len`), and `is_separator_regex` (set to `False`)

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=200,
                                               length_function=len,
                                               is_separator_regex=False)

To embed a set of documents in the vector database you can use the `add_documents()` method with the `documents` parameter set to a list of the document texts and `ids` set to a list of unique identifiers.

2.3 (15 pts) Loop over all the `.txt` files in the `docs` subdirectory (hint: use the pathlib.Path.glob() method). For each file do the following:
 - use the `text_splitter` instance you created above and call the `split_text()` method passing in the file contents to create a list of text chunks
 - for each text chunk create an instance of the `Document` class (from `langchain_core.documents`) setting the `page_content` param to the chunk and the `metatdata` parameter to a `dict` having keys `source` and `title`, both set to the filenaame. Add the Document object to a list
 - create a list of uuids the same length as the list of `Document` objects using the `uuid4()` function (from the `uuid` module). Note: You will need to convert the uuids to strings.
 - call the `add_documents()` method on the vector database instance and set the `documents` parameter to the list of `Document` objects and the `ids` param to the list of uuids you created

In [12]:
from pathlib import Path
from langchain_core.documents import Document
import uuid

documents = []

p = Path('./docs')
for file in p.glob('*.txt'):
    with file.open("r", encoding="utf-8") as f:
        content = f.read()
        chunks = text_splitter.split_text(content)
        for chunk in chunks:
            documents.append(Document(page_content=chunk, metadata={"source": file.name, "title": file.name}))

doc_ids = [str(uuid.uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=doc_ids)

['2afee940-2ebe-4b75-b726-02741e2e4d6d',
 'baa89d86-e477-4db7-9510-bee59caac337',
 '00d0a4ee-61b4-45e0-a04b-fe651b2d559d',
 '6db5da2e-a6b0-4416-859b-4de04e96eb71',
 'ec75787f-0b38-415c-a68d-00c4657afe65',
 '05b80695-6f85-43d9-a4ad-7874f6ddc2e2',
 'c0207951-5fe9-4530-bf4a-69fa9516ee25',
 '6628157e-3e7f-474d-a993-4a442c5f7daa',
 '05a67e65-d3ec-4bac-8d2f-cf23d213ccab',
 'eae691fb-c114-47b2-bbc9-1894e08f8a1d',
 'c66f8745-51cf-4215-b35e-6b2962450306',
 '17406796-d93a-495f-b303-134972984e3c',
 '5b5b0dbe-3d60-481c-aefc-7a57e1287ad4',
 '17ade884-a245-48c0-8992-4b4375d27430',
 'e0e60b07-0bc8-4491-b93f-abbdfc215101',
 '6ac0eb1d-9a29-41bc-8e78-4dca0c49a134',
 'ef10aaf3-9472-46d9-a057-ed731e3342c1',
 '8beba190-5e0e-4e54-8e15-f4f93fe169d4',
 '29cd5ec9-8cf6-4d7f-8d48-a16d19961d50',
 '223193bd-49e9-4687-863d-0e9f43d2b5ae',
 '2edd9293-b413-4ee3-966a-ed3583d2e6ee',
 '4735610d-179f-40f1-87d3-69dbbb1a271b',
 '943d1dba-6549-435b-9213-c4097c872aa4',
 '885823f4-c01c-4120-bdab-73759b0eb8f4',
 '1fdf3ee8-bf39-

2.4 (5 pts) Run the queries below to test the search function of your vector store (you may need to adjust the variable names to match yours)

In [13]:
for doc in vector_store.similarity_search_by_vector(embedding=embed.embed_query("seed grant"), k=2):
    print(f"*** {doc.page_content} [{doc.metadata}]\n")

*** New grant fuels research on energy–
water resilience
December 23, 2025

Irene Cionni, a senior research scholar in Boise State’s Department of Geosciences, has been
awarded a seed grant from I-CREWS (Idaho Community-engaged Resilience for Energy-Water
Systems), a National Science Foundation EPSCoR program. The seed grant enables her to advance
studies on compound energy drought — a weather phenomenon that poses significant challenges to
Idaho’s energy-water systems.

Irene Cionni, Ph.D., Senior Research Scholar in Boise State’s Department of Geosciences

Dunkelflauten: a German phrase for an Idaho
energy challenge
Compound energy drought occurs when multiple renewable energy sources experience [{'title': 'New grant fuels research on energy–water resilience - Boise State News.txt', 'source': 'New grant fuels research on energy–water resilience - Boise State News.txt'}]

*** New grant fuels research on energy–
water resilience
December 23, 2025

Irene Cionni, a senior research schola

2.5 (10 pts) Now that you know how to create embeddings and a vector store for searching, implement the function `rag(query, k=3)` in the `b1lib.py` module. The `query` parameter will be a string query that you will need to embed and pass to the vector database `similarity_search_by_vector()` function (refer to the unit test above). You will first need to create global instances of the `OllamaEmbeddings` and the `Chroma` objects for your `rag()` function to use (make sure you set the `.env` file to use the values you created above). Then use the unit test below to show it working. Notice for each RAG document retrieved the `page_content` property holds the actual text value. (Useful later on.)

In [14]:
def rag(query, k=3):
    query_embed = embed.embed_query(query)
    docs = vector_store.similarity_search_by_vector(embedding=query_embed, k=k)
    return docs

### 3. The Chat Interface

The last piece is to enhance the chat interface you created in Lab 1 and include the rag data in the context window.

3.1 (5 pts) Copy your `lab1lib.py` from Lab 1. Make sure your `.env` file is set to use a local ollama generative model (such as `phi3.5` or `nemotron-3-nano`). Import the `chatlib` module and create an instance of the model by calling the `get_chat_model()` helper function and name it `llm`.

In [15]:
import chatlib
from dotenv import load_dotenv
load_dotenv('sample.env')
llm = chatlib.get_chat_model()

Using ollama with model phi3.5


3.2 (5 pts) Call the `invoke()` method on your `ChatModel` instance passing in a list of messages with a single message ```"What recent seed grants were awarded at Boise State?"```. You will need to wrap the message in an instance of the `HumanMessage` class (set the `content` parameter in the constructor to the message). Note: the `HumanMessage` class can be imported from the `langchain_core.messages` module.

In [16]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content='What recent seed grants were awarded at Boise State?')]
response = llm.chat(messages)
print(response)

content='As of my knowledge cutoff date in early 2023, I don\'thy have real-time information or the ability to browse the internet for up-to-date news on specific events such as current grant awards. However, if you are interested in recent seed grants awarded at Boise State University (BSU), here is a general approach that can help guide your search:\n\n1. **University Website** - Visit BSU\'s official website and look for sections related to research centers or the Office of Research & Grants where they may list their most recent grant awards, including seed grants if applicable. They often have announcements about funded projects on institutional news sites like a dedicated page under "news" or in press releases posted regularly.\n\n2. **Research Repository** - Universities typically maintain databases that can be searched for research articles and reports which may include details of grants awarded, including seed-funding opportunities provided to students or faculty members: https

---
The LLM cannot give recent information becuase it was trained on older data. RAG to the rescue!

3.3 (15 pts) Modify your `chat()` function in `lab1lib` to inject RAG data into the model context window for each request:
1. Add a `rag_query` parameter (default `None`)
2. If `rag_query` is not `None` call into `b1lib::rag()` to get the RAG data
3. Join the results from `rag()` into a single string using the `page_content` property of each returned item (add any formatting/separators if you wish)
4. Create a `SystemMessage` instance using your rag data string
5. Prepend your `SystemMessage` to the list of messages before calling the `chat()` method on your `ChatModel`.
  
Once your implemenation is working, import your `lab1lib` and run the same query as you did in 3.2 but now using your `lab1lib::chat()` function. Set the `chat_id` parameter to an arbitrary string, the `user` parameter to the query (from 3.2), and the `rag_query` parameter to `seed grant`.

In [22]:
from lab1lib import chat
chat(chat_id="random", user_message='What recent seed grants were awarded at Boise State?', rag_query="seed grant")

DEBUG: Found 3 documents for query


3.4 (5 pts) Your `chat()` function updates your internal chat data scructure. To see the output you need to call the `get_chat_html()` function in `lab1lib` and call it with the `chat_id` you used above.

In [23]:
from lab1lib import get_chat_html
get_chat_html(chat_id='random')

Metadata for message: {'token_usage': {'prompt_tokens': 605}}
Metadata for message: {'model': 'phi3.5', 'created_at': '2026-02-27T06:05:42.060562602Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1052323586, 'load_duration': 17762277, 'prompt_eval_count': 605, 'prompt_eval_duration': 73894294, 'eval_count': 211, 'eval_duration': 897222662, 'logprobs': None, 'model_name': 'phi3.5', 'model_provider': 'ollama'}
Metadata for message: {'token_usage': {'prompt_tokens': 839}}
Metadata for message: {'model': 'phi3.5', 'created_at': '2026-02-27T06:08:07.96548135Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4174408776, 'load_duration': 1963263789, 'prompt_eval_count': 839, 'prompt_eval_duration': 88683583, 'eval_count': 446, 'eval_duration': 1996312272, 'logprobs': None, 'model_name': 'phi3.5', 'model_provider': 'ollama'}
Metadata for message: {}
Metadata for message: {'token_usage': {'prompt_tokens': 1322}}
Metadata for message: {'model': 'phi3.5', 'created_at': '2026-02

'<div class=\'user\'> <p>What recent seed grants were awarded at Boise State?</p> <div class=\'info\'><div> <i class=\'fas fa-coins text-warning\'></i> 605</div></div> </div><div class=\'assistant\'> <p>As of my knowledge cutoff date in early 2023, I do not have access to real-time databases or the ability to provide current news updates. However, based on your provided context:</p>\n<p>One notable example is the seed grant given by I-CREWS (Idaho Community-engaged Resilience for Energy-Water Systems) awarded to Dr. Irene Cionni from Boise State\'s Department of Geosciences. The purpose was to further her research on compound energy drought, which can create substantial stress within Idaho’s integrated systems combining renewable energies and water management due to varying weather patterns affecting different sources simultaneously or sequentially.</p>\n<p>For the most recent information about seed grants awarded at Boise State University (BSU), please visit BSU\'s official website un

### 4. Putting it All Together

Finally, copy your `app.py` and `templates` from Lab 1. Add a RAG checkbox to the input controls in your `templates/form.html` file. In your `app.py` `submit()` function, test if RAG is checked and if it is set the `rag_query` parameter in your call to `lab1.chat()` to the user query.

Run the Flask app by going to a terminal and executing `python app.py`. Note the port number and use your browser to load your web interface, e.g. `http://127.0.0.1:5000`.

4.1 (10 pts) Look at the files in the `docs` subdirectory and try asking your model specific questions with and without the "Use Rag" input checked. Include a screenshot in your submission of at least one question with/without RAG showing the model cannot answer it unless it has the RAG context. In the cell below give a brief analysis of your RAG pipeline and its benefits and limitations.

In [24]:
chat(chat_id="without_rag", user_message='Tell me some information about the Silver Medallion Awards')

In [25]:
get_chat_html(chat_id='without_rag')

Metadata for message: {'token_usage': {'prompt_tokens': 32}}
Metadata for message: {'model': 'phi3.5', 'created_at': '2026-02-27T06:36:57.368442388Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6883443330, 'load_duration': 4659412786, 'prompt_eval_count': 32, 'prompt_eval_duration': 24022697, 'eval_count': 514, 'eval_duration': 2059478103, 'logprobs': None, 'model_name': 'phi3.5', 'model_provider': 'ollama'}


'<div class=\'user\'> <p>Can you tell me some information about the winners of the Silver Medallion Awards in 2025</p> <div class=\'info\'><div> <i class=\'fas fa-coins text-warning\'></i> 32</div></div> </div><div class=\'assistant\'> <p>As of my knowledge cutoff date in March 2 extrndary, I do not have specific real-time or future event data such as details on who won a hypothetical "Silver Medallion Award" in 2025. If the Silver Medallion Awards are an actual competition that took place after this time, you may need to look up recent news articles, official announcements from organizations hosting these awards, or their dedicated websites for accurate information about winners and events related to them.</p>\n<p>Here\'s how I can assist if such data is available:</p>\n<ol>\n<li>\n<p><strong>Research Updates</strong>: Visit credible sources like award-giving organization’s official website where they might have a news section with updates on the latest ceremonies or winners lists pub

In [28]:
rag_query = 'Tell me some information about the Silver Medallion Awards'
chat(chat_id="with_rag", user_message=rag_query, rag_query=rag_query)

DEBUG: Found 3 documents for query


In [31]:
get_chat_html(chat_id='with_rag')

Metadata for message: {'token_usage': {'prompt_tokens': 818}}
Metadata for message: {'model': 'phi3.5', 'created_at': '2026-02-27T06:40:32.709119066Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1725987984, 'load_duration': 18566424, 'prompt_eval_count': 818, 'prompt_eval_duration': 12523965, 'eval_count': 357, 'eval_duration': 1593154216, 'logprobs': None, 'model_name': 'phi3.5', 'model_provider': 'ollama'}


"<div class='user'> <p>Tell me some information about the Silver Medallion Awards</p> <div class='info'><div> <i class='fas fa-coins text-warning'></i> 818</div></div> </div><div class='assistant'> <p>The Silver Medallion Awards at Boise State University, established in 1969 by President John Barnes, are prestigious accolades that recognize individuals who have provided sustained service or made significant contributions to advancing the university's mission over time. Recipients of these awards demonstrate a deep commitment and impactful involvement within various aspects of Boise State University’s community—including academics, student life, athletics, campus initiatives, fundraising efforts, alumni relations, or other relevant areas that foster the university's goals.</p>\n<p>These recipients are celebrated at a special commencement ceremony on December 20th each year where they receive their Silver Medallion Award and recognition from faculty, staff members, students, as well as g

My current RAG pipeline has the obvious limitation of only having access to 9 documents.  It also hasn't been the most accurrate thing in the world when it comes to generation.  It also only returns the top 3 document chunks which might not be enough context.  I had a lot of trouble trying to get the UI interface to work (the rag_query wasn't ever being sent so that's something I have to work on.  The benefits of this RAG pipeline is that its fully open source and easy to add more documents to.  It's also fully locally run so there isn't much security risk as well.

### Submission Instructions

Be sure to ***SAVE YOUR WORK***!  

Then submit the following to Canvas:
1. This notebook (.ipynb) file (do not clear the cell outputs) using the original name - do not rename it!
2. A screenshot as described in 4.1
3. Your b1lib.py implementation